In [1]:
%cd ..

/home/dmoreno/pipeline_v4_final/pipeline/training/stamp_classifier/data_acquisition/rubin


In [2]:
import pandas as pd
import numpy as np
import glob
import os

In [ ]:
PATH_PARTITION_REAL_DATA = '/home/dmoreno/pipeline_v4_final/pipeline/training/stamp_classifier/data_acquisition/rubin/data/processed/partitions/partitions_trainSN_valSN_real_eval_firstStamp_v1/partitions.parquet'
PATH_SIMULATED_DATA = '/home/dmoreno/pipeline_v4_final/pipeline/training/stamp_classifier/data_acquisition/rubin/data/simulated_data/synthetic_total_SN.pkl'


PATH_TO_SAVE_MIXED_PARTITIONS = '/home/dmoreno/pipeline_v4_final/pipeline/training/stamp_classifier/data_acquisition/rubin/data/processed/partitions/partitions_trainSN_mixed_valSN_real_eval_firstStamp_v1/partitions.parquet'

In [12]:
real_partitions = pd.read_parquet(PATH_PARTITION_REAL_DATA)
real_partitions

,oid,oid_kept,measurement_id,band,ra,dec,mjd,class,partition,dataset_origin,synth_file_path
0,169298432645136566,169298432645136566,169298432645136566,i,8.905992,-44.103805,60924.330414,AGN,training_0,real,NaN
1,169298432645136566,169298432645136566,169298432645136566,i,8.905992,-44.103805,60924.330414,AGN,training_1,real,NaN
2,169298432645136566,169298432645136566,169298432645136566,i,8.905992,-44.103805,60924.330414,AGN,training_2,real,NaN
3,169298432645136566,169298432645136566,169298432645136566,i,8.905992,-44.103805,60924.330414,AGN,validation_3,real,NaN
4,169298432645136566,169298432645136566,169298432645136566,i,8.905992,-44.103805,60924.330414,AGN,training_4,real,NaN
...,...,...,...,...,...,...,...,...,...,...,...
30039,169623886116683891,169623886116683891,169623886116683891,r,7.626684,-43.558806,60998.223125,bogus,validation_0,real,NaN
30040,169623886116683891,169623886116683891,169623886116683891,r,7.626684,-43.558806,60998.223125,bogus,training_1,real,NaN
30041,169623886116683891,169623886116683891,169623886116683891,r,7.626684,-43.558806,60998.223125,bogus,training_2,real,NaN
30042,169623886116683891,169623886116683891,169623886116683891,r,7.626684,-43.558806,60998.223125,bogus,training_3,real,NaN


In [13]:
real_partitions.dataset_origin.unique()

array(['real'], dtype=object)

In [14]:
# Total de estampillas en cada subset

conteo = real_partitions.groupby(['partition', 'class']).size().reset_index(name='count')
conteo_pivot = conteo.pivot(index='partition', columns='class', values='count').fillna(0)
print(conteo_pivot)

class          AGN   SN   VS  asteroid  bogus
partition                                    
test           129   54  174       419    423
training_0    1098  111  743      1459   1405
training_1    1068  113  745      1486   1393
training_2    1082  114  738      1472   1392
training_3    1061  110  737      1454   1401
training_4    1074  112  728      1485   1394
validation_0    81   23  131       380    338
validation_1   111   21  129       353    350
validation_2    97   20  136       367    351
validation_3   118   24  137       385    342
validation_4   105   22  146       354    349


In [15]:
# Total de objetos en cada subset

partitions_real_first = real_partitions.sort_values(['oid', 'mjd']).groupby(['oid_kept', 'partition']).first().reset_index()
conteo = partitions_real_first.groupby(['partition', 'class']).size().reset_index(name='count')
conteo_pivot = conteo.pivot(index='partition', columns='class', values='count').fillna(0)
print(conteo_pivot)

class         AGN  SN   VS  asteroid  bogus
partition                                  
test           41  19  112       200    397
training_0    204  15  410       640   1284
training_1    204  15  410       640   1284
training_2    203  15  410       640   1285
training_3    204  15  410       640   1285
training_4    204  15  410       640   1285
validation_0   33   8   90       160    319
validation_1   33   8   90       160    319
validation_2   34   8   90       160    318
validation_3   33   8   90       160    318
validation_4   33   8   90       160    318


In [27]:
import numpy as np

df_sinteticos = pd.read_pickle(PATH_SIMULATED_DATA)
df_sinteticos = df_sinteticos.rename(columns={
    'gal_objectId': 'oid',
    'diff_sourceId': 'measurement_id'
    })

df_sinteticos['class'] = 'SN'
df_sinteticos = df_sinteticos[['oid', 'measurement_id', 'band', 'SN_ellip_dist', 'class', 'synth_file_path']]
df_sinteticos['dataset_origin'] = 'synthetic'

# Ffroster no genero estas columnas
df_sinteticos['ra'] = np.nan
df_sinteticos['dec'] = np.nan
df_sinteticos['mjd'] = np.nan

# Todos las estampillas
df_sinteticos['oid_kept'] = df_sinteticos['oid']
df_sinteticos = df_sinteticos.sort_values(['oid_kept', 'SN_ellip_dist'])
df_sinteticos = df_sinteticos[df_sinteticos['SN_ellip_dist'] <= 0.5]
df_sinteticos_all = df_sinteticos.copy()

# Solo una estampilla por galaxia para hacer las divisiones
df_sinteticos = df_sinteticos.groupby('oid_kept').first().reset_index()
df_sinteticos

,oid_kept,oid,measurement_id,band,SN_ellip_dist,class,synth_file_path,dataset_origin,ra,dec,mjd
0,609780764988422794,609780764988422794,648372729770148027,g,0.08,SN,609780764988422794_g_2024120900332_6_648372729...,synthetic,NaN,NaN,NaN
1,609780833707896898,609780833707896898,609782139377943160,g,0.06,SN,609780833707896898_g_2024112900266_0_609782139...,synthetic,NaN,NaN,NaN
2,609780833707898122,609780833707898122,592913981740417050,g,0.20,SN,609780833707898122_g_2024120300123_0_592913981...,synthetic,NaN,NaN,NaN
3,609780833707900556,609780833707900556,648373485684392217,i,0.25,SN,609780833707900556_i_2024112900274_6_648373485...,synthetic,NaN,NaN,NaN
4,609780833707901751,609780833707901751,648372798489624918,g,0.07,SN,609780833707901751_g_2024120900332_3_648372798...,synthetic,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...
163,611256515751335758,611256515751335758,609789629800907672,z,0.20,SN,611256515751335758_z_2024120700332_2_609789629...,synthetic,NaN,NaN,NaN
164,611256584470805653,611256584470805653,592914119179370901,y,0.17,SN,611256584470805653_y_2024112000223_2_592914119...,synthetic,NaN,NaN,NaN
165,611256584470807624,611256584470807624,648366407578288502,z,0.50,SN,611256584470807624_z_2024120700282_2_648366407...,synthetic,NaN,NaN,NaN
166,611256584470808273,611256584470808273,609782139377943160,g,0.05,SN,611256584470808273_g_2024120300165_2_609782139...,synthetic,NaN,NaN,NaN


In [28]:
print(f'Numero de estampillas: {df_sinteticos_all.shape}')

Numero de estampillas: (2089, 11)


In [30]:
print(f'Numero de galaxias: {df_sinteticos.shape}')

# Pueuden haber varios objetos en una galaxia, pero para que no haya data leakage, mantenemos objetos
# de una misma galaxia en el mismo subset

Numero de galaxias: (168, 11)


In [32]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, StratifiedKFold

# ==============================================================================
# 2. LÓGICA PARA CREAR EL ARCHIVO ACUMULADO
# ==============================================================================
print("--- 2. Generando el DataFrame acumulado con particiones explícitas ---")

# --- PASO A: Identificar objetos únicos para la división ---
unique_objects = df_sinteticos[['oid_kept', 'class']].drop_duplicates().reset_index(drop=True)

# --- PASO B: Separar el conjunto de TEST (20%) ---
train_val_objects, test_objects = train_test_split(
    unique_objects,
    test_size=0.20,
    random_state=42,
    stratify=unique_objects['class']
)
test_oids = set(test_objects['oid_kept'])

# --- PASO C: Asignar un número de fold (0 a 4) a cada objeto de train/validation ---
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
train_val_objects = train_val_objects.copy()
# Creamos un mapa de oid -> número de fold de validación
oid_to_fold_map = {}
for fold_num, (_, val_idx) in enumerate(skf.split(train_val_objects, train_val_objects['class'])):
    val_oids = train_val_objects.iloc[val_idx]['oid_kept']
    for oid in val_oids:
        oid_to_fold_map[oid] = fold_num

# --- PASO D: Construir la lista de DataFrames para cada partición y acumular ---
all_partitions_list = []

# 1. Agregar el conjunto de TEST (se agrega una sola vez)
test_df = df_sinteticos[df_sinteticos['oid_kept'].isin(test_oids)].copy()
test_df['partition'] = 'test'
all_partitions_list.append(test_df)

# 2. Iterar por cada fold para crear los conjuntos de training y validation
for i in range(5):
    # Identificar los OIDs para este fold
    validation_oids_fold_i = {oid for oid, fold in oid_to_fold_map.items() if fold == i}
    training_oids_fold_i = {oid for oid, fold in oid_to_fold_map.items() if fold != i}
    
    # Crear el DataFrame de VALIDACIÓN para el fold i
    validation_df_fold_i = df_sinteticos[df_sinteticos['oid_kept'].isin(validation_oids_fold_i)].copy()
    validation_df_fold_i['partition'] = f'validation_{i}'
    all_partitions_list.append(validation_df_fold_i)
    
    # Crear el DataFrame de ENTRENAMIENTO para el fold i
    training_df_fold_i = df_sinteticos[df_sinteticos['oid_kept'].isin(training_oids_fold_i)].copy()
    training_df_fold_i['partition'] = f'training_{i}'
    all_partitions_list.append(training_df_fold_i)
    
# --- PASO E: Concatenar todos los DataFrames en uno solo ---
final_df = pd.concat(all_partitions_list, ignore_index=True)
final_df

--- 2. Generando el DataFrame acumulado con particiones explícitas ---


,oid_kept,oid,measurement_id,band,SN_ellip_dist,class,synth_file_path,dataset_origin,ra,dec,mjd,partition
0,609780833707896898,609780833707896898,609782139377943160,g,0.06,SN,609780833707896898_g_2024112900266_0_609782139...,synthetic,NaN,NaN,NaN,test
1,609782139377951690,609782139377951690,648366407578288506,z,0.33,SN,609782139377951690_z_2024120700315_4_648366407...,synthetic,NaN,NaN,NaN,test
2,609782276816909559,609782276816909559,648364964469277207,i,0.33,SN,609782276816909559_i_2024112900226_0_648364964...,synthetic,NaN,NaN,NaN,test
3,609782276816911518,609782276816911518,611255003922831024,g,0.33,SN,609782276816911518_g_2024120300123_1_611255003...,synthetic,NaN,NaN,NaN,test
4,609788255411390363,609788255411390363,648365033188753746,g,0.05,SN,609788255411390363_g_2024113000171_5_648365033...,synthetic,NaN,NaN,NaN,test
...,...,...,...,...,...,...,...,...,...,...,...,...
699,611256447031856263,611256447031856263,648368125565206567,g,0.12,SN,611256447031856263_g_2024120300150_2_648368125...,synthetic,NaN,NaN,NaN,training_4
700,611256515751328054,611256515751328054,592913294545649750,g,0.50,SN,611256515751328054_g_2024112900233_2_592913294...,synthetic,NaN,NaN,NaN,training_4
701,611256515751329782,611256515751329782,609789629800907672,z,0.25,SN,611256515751329782_z_2024111700145_0_609789629...,synthetic,NaN,NaN,NaN,training_4
702,611256584470805653,611256584470805653,592914119179370901,y,0.17,SN,611256584470805653_y_2024112000223_2_592914119...,synthetic,NaN,NaN,NaN,training_4


In [37]:
conteo = final_df.groupby(['partition', 'class']).size().reset_index(name='count')
conteo_pivot = conteo.pivot(index='partition', columns='class', values='count').fillna(0)
print('Esto muestra la cantidad de galaxias que quedaron en cada subset:\n')
print(conteo_pivot)

Esto muestra la cantidad de galaxias que quedaron en cada subset:

class          SN
partition        
test           34
training_0    107
training_1    107
training_2    107
training_3    107
training_4    108
validation_0   27
validation_1   27
validation_2   27
validation_3   27
validation_4   26


In [34]:
partition_map = final_df[['oid_kept', 'class', 'partition']].drop_duplicates()

print("\n--- Mapa de Partición Limpio (lo que se usará para el merge) ---")
print(partition_map)
print(f"\nForma: {partition_map.shape}")
print("-" * 50)


# --- Paso 2: Unir (Merge) los datos completos con el mapa de partición ---
# Usamos un 'left' merge: empezamos con `objs` (todos los datos) y le añadimos
# la columna 'partition' del mapa, coincidiendo por `oid_kept` y `class`.
# De esta forma, cada observación en `objs` recibirá la(s) etiqueta(s) de partición
# que le corresponde a su objeto.

final_dataset_partitioned = pd.merge(
    df_sinteticos_all,
    partition_map,
    on=['oid_kept', 'class'], # Columnas clave para la unión
    how='left'
)

final_dataset_partitioned


--- Mapa de Partición Limpio (lo que se usará para el merge) ---
               oid_kept class   partition
0    609780833707896898    SN        test
1    609782139377951690    SN        test
2    609782276816909559    SN        test
3    609782276816911518    SN        test
4    609788255411390363    SN        test
..                  ...   ...         ...
699  611256447031856263    SN  training_4
700  611256515751328054    SN  training_4
701  611256515751329782    SN  training_4
702  611256584470805653    SN  training_4
703  611256584470808273    SN  training_4

[704 rows x 3 columns]

Forma: (704, 3)
--------------------------------------------------


,oid,measurement_id,band,SN_ellip_dist,class,synth_file_path,dataset_origin,ra,dec,mjd,oid_kept,partition
0,609780764988422794,648372729770148027,g,0.08,SN,609780764988422794_g_2024120900332_6_648372729...,synthetic,NaN,NaN,NaN,609780764988422794,training_0
1,609780764988422794,648372729770148027,g,0.08,SN,609780764988422794_g_2024120900332_6_648372729...,synthetic,NaN,NaN,NaN,609780764988422794,validation_1
2,609780764988422794,648372729770148027,g,0.08,SN,609780764988422794_g_2024120900332_6_648372729...,synthetic,NaN,NaN,NaN,609780764988422794,training_2
3,609780764988422794,648372729770148027,g,0.08,SN,609780764988422794_g_2024120900332_6_648372729...,synthetic,NaN,NaN,NaN,609780764988422794,training_3
4,609780764988422794,648372729770148027,g,0.08,SN,609780764988422794_g_2024120900332_6_648372729...,synthetic,NaN,NaN,NaN,609780764988422794,training_4
...,...,...,...,...,...,...,...,...,...,...,...,...
8656,611257134226611503,648367026053579334,i,0.37,SN,611257134226611503_i_2024112900276_2_648367026...,synthetic,NaN,NaN,NaN,611257134226611503,test
8657,611257134226611503,614439120877388319,u,0.40,SN,611257134226611503_u_2024120100160_0_614439120...,synthetic,NaN,NaN,NaN,611257134226611503,test
8658,611257134226611503,648367026053579334,i,0.42,SN,611257134226611503_i_2024112900276_2_648367026...,synthetic,NaN,NaN,NaN,611257134226611503,test
8659,611257134226611503,648372111294857345,i,0.47,SN,611257134226611503_i_2024112900276_2_648372111...,synthetic,NaN,NaN,NaN,611257134226611503,test


In [35]:
final_column_order = [
    'oid', 'oid_kept', 'measurement_id', 'band', 'ra', 'dec', 'mjd',
    'class', 'partition', 'dataset_origin', 'synth_file_path'
]
final_dataset_partitioned = final_dataset_partitioned[final_column_order]
final_dataset_partitioned

,oid,oid_kept,measurement_id,band,ra,dec,mjd,class,partition,dataset_origin,synth_file_path
0,609780764988422794,609780764988422794,648372729770148027,g,NaN,NaN,NaN,SN,training_0,synthetic,609780764988422794_g_2024120900332_6_648372729...
1,609780764988422794,609780764988422794,648372729770148027,g,NaN,NaN,NaN,SN,validation_1,synthetic,609780764988422794_g_2024120900332_6_648372729...
2,609780764988422794,609780764988422794,648372729770148027,g,NaN,NaN,NaN,SN,training_2,synthetic,609780764988422794_g_2024120900332_6_648372729...
3,609780764988422794,609780764988422794,648372729770148027,g,NaN,NaN,NaN,SN,training_3,synthetic,609780764988422794_g_2024120900332_6_648372729...
4,609780764988422794,609780764988422794,648372729770148027,g,NaN,NaN,NaN,SN,training_4,synthetic,609780764988422794_g_2024120900332_6_648372729...
...,...,...,...,...,...,...,...,...,...,...,...
8656,611257134226611503,611257134226611503,648367026053579334,i,NaN,NaN,NaN,SN,test,synthetic,611257134226611503_i_2024112900276_2_648367026...
8657,611257134226611503,611257134226611503,614439120877388319,u,NaN,NaN,NaN,SN,test,synthetic,611257134226611503_u_2024120100160_0_614439120...
8658,611257134226611503,611257134226611503,648367026053579334,i,NaN,NaN,NaN,SN,test,synthetic,611257134226611503_i_2024112900276_2_648367026...
8659,611257134226611503,611257134226611503,648372111294857345,i,NaN,NaN,NaN,SN,test,synthetic,611257134226611503_i_2024112900276_2_648372111...


In [38]:
conteo = final_dataset_partitioned.groupby(['partition', 'class']).size().reset_index(name='count')
conteo_pivot = conteo.pivot(index='partition', columns='class', values='count').fillna(0)
print('Esto muestra la cantidad de estampillas de una misma galaxia que quedaron en cada subset:\n')
print(conteo_pivot)

Esto muestra la cantidad de estampillas de una misma galaxia que quedaron en cada subset:

class           SN
partition         
test           446
training_0    1326
training_1    1341
training_2    1281
training_3    1294
training_4    1330
validation_0   317
validation_1   302
validation_2   362
validation_3   349
validation_4   313


In [43]:
# Consideramos dejar supernovas sinteticas solo en entrenamiento y en test para evaluar como funciona en los datos sinteticos, pero
# la evaluación se sigue realizando sobre los datos reales de test nomas. Aunque igual se puede considerar tener supernovas sinteticas en validación
# depende lo que una prefiera.

# Primero, identifica qué particiones son de validación
validation_partitions = ['validation_0', 'validation_1', 'validation_2', 'validation_3', 'validation_4']

# Crea una máscara para identificar las filas que NO deben ser removidas
# Queremos mantener:
# 1. Todas las filas que NO sean de validación (otras particiones)
# 2. Las filas de validación que NO sean supernovas

mask = ~(
    (final_dataset_partitioned['partition'].isin(validation_partitions)) & 
    (final_dataset_partitioned['class'] == 'SN')
)

# Aplica la máscara para obtener el dataframe filtrado
final_df_filtered = final_dataset_partitioned[mask].copy()

conteo = final_df_filtered.groupby(['partition', 'class']).size().reset_index(name='count')
conteo_pivot = conteo.pivot(index='partition', columns='class', values='count').fillna(0)
print(conteo_pivot)

class         SN
partition       
test         446
training_0  1326
training_1  1341
training_2  1281
training_3  1294
training_4  1330


In [44]:
partitions_mixed = pd.concat([real_partitions, final_df_filtered]).reset_index(drop=True)
partitions_mixed

,oid,oid_kept,measurement_id,band,ra,dec,mjd,class,partition,dataset_origin,synth_file_path
0,169298432645136566,169298432645136566,169298432645136566,i,8.905992,-44.103805,60924.330414,AGN,training_0,real,NaN
1,169298432645136566,169298432645136566,169298432645136566,i,8.905992,-44.103805,60924.330414,AGN,training_1,real,NaN
2,169298432645136566,169298432645136566,169298432645136566,i,8.905992,-44.103805,60924.330414,AGN,training_2,real,NaN
3,169298432645136566,169298432645136566,169298432645136566,i,8.905992,-44.103805,60924.330414,AGN,validation_3,real,NaN
4,169298432645136566,169298432645136566,169298432645136566,i,8.905992,-44.103805,60924.330414,AGN,training_4,real,NaN
...,...,...,...,...,...,...,...,...,...,...,...
37057,611257134226611503,611257134226611503,648367026053579334,i,<NA>,<NA>,<NA>,SN,test,synthetic,611257134226611503_i_2024112900276_2_648367026...
37058,611257134226611503,611257134226611503,614439120877388319,u,<NA>,<NA>,<NA>,SN,test,synthetic,611257134226611503_u_2024120100160_0_614439120...
37059,611257134226611503,611257134226611503,648367026053579334,i,<NA>,<NA>,<NA>,SN,test,synthetic,611257134226611503_i_2024112900276_2_648367026...
37060,611257134226611503,611257134226611503,648372111294857345,i,<NA>,<NA>,<NA>,SN,test,synthetic,611257134226611503_i_2024112900276_2_648372111...


In [45]:
conteo = partitions_mixed.groupby(['partition', 'class']).size().reset_index(name='count')
conteo_pivot = conteo.pivot(index='partition', columns='class', values='count').fillna(0)
print(conteo_pivot)

class          AGN    SN   VS  asteroid  bogus
partition                                     
test           129   500  174       419    423
training_0    1098  1437  743      1459   1405
training_1    1068  1454  745      1486   1393
training_2    1082  1395  738      1472   1392
training_3    1061  1404  737      1454   1401
training_4    1074  1442  728      1485   1394
validation_0    81    23  131       380    338
validation_1   111    21  129       353    350
validation_2    97    20  136       367    351
validation_3   118    24  137       385    342
validation_4   105    22  146       354    349


In [46]:
partitions_mixed_first_rep = partitions_mixed.sort_values(['oid', 'mjd']).groupby(['oid', 'partition']).first().reset_index()
conteo = partitions_mixed_first_rep.groupby(['partition', 'class']).size().reset_index(name='count')
conteo_pivot = conteo.pivot(index='partition', columns='class', values='count').fillna(0)
print(conteo_pivot)

class         AGN   SN   VS  asteroid  bogus
partition                                   
test           50   57  119       200    397
training_0    433  143  497       640   1284
training_1    423  142  500       640   1284
training_2    426  143  496       640   1285
training_3    428  142  499       640   1285
training_4    425  144  497       640   1285
validation_0   39    8   94       160    319
validation_1   49    9   91       160    319
validation_2   46    8   95       160    318
validation_3   44    9   92       160    318
validation_4   47    8   94       160    318


In [ ]:
# Convertir la columna oid a string explícitamente
partitions_mixed['oid'] = partitions_mixed['oid'].astype('int64')
partitions_mixed['measurement_id'] = partitions_mixed['measurement_id'].astype('int64')
partitions_mixed['oid_kept'] = partitions_mixed['oid_kept'].astype('int64')

partitions_mixed.to_parquet(PATH_TO_SAVE_MIXED_PARTITIONS)